In [2]:
from dotenv import load_dotenv
load_dotenv()
import os 

PROJECT_ABSOLUTE_PATH = os.getenv("PROJECT_ABSOLUTE_PATH")

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-20b',temperature=0.1)

In [6]:
from fastmcp import Client, ClientGroup
from fastmcp.client.transports import StdioTransport
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage

In [7]:
exchange_rate_transport = StdioTransport(
    command=f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/.venv/bin/python",
    args=[
        f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/week-6/practice/exchange_rate_conversion_server.py"
    ]
)

exchange_rate_client = Client(transport=exchange_rate_transport)

In [8]:
manim_transport = StdioTransport(
    command=f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/.venv/bin/python",
    args=[
        f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/week-6/practice/manim_server.py"
    ],
    env={
        "MANIM_EXECUTABLE": f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/.venv/bin/manim"
    }
)

manim_client = Client(transport=manim_transport)

In [18]:
clients = ClientGroup({
    "manim": manim_client,
    "exchange_rate": exchange_rate_client,
    "fast_crawl_mcp": Client("https://mcp.firecrawl.dev/v2/mcp")
})

In [19]:
async with MCPAdapter(clients) as adapter:
    tools = await adapter.list_tools()

    print("=== TOOLS ===")
    for tool in tools:
        print(tool.name)

=== TOOLS ===
manim_execute_manim_code
manim_cleanup_manim_temp_dir
exchange_rate_get_conversion_factor
exchange_rate_convert
fast_crawl_mcp_firecrawl_scrape
fast_crawl_mcp_firecrawl_search
fast_crawl_mcp_firecrawl_parse


In [32]:
async with MCPAdapter(clients) as adapter:
    tools = await adapter.list_tools()

    agent = create_agent(
        model=llm,
        tools=tools,
    )

    query = HumanMessage('What is 20 dollars equivalent to indian rupee?')
    
    initial_state = {"messages": [query]}
    result = await agent.ainvoke(initial_state)
    # print(result)
    last_message = result["messages"][-1]
    print(last_message.content)

$20 USD is roughly equivalent to **₹1,919** (Indian rupees).


In [ ]:
async with MCPAdapter(clients) as adapter:

    tools = await adapter.list_tools()

    agent = create_agent(
            model=llm,
            tools=tools,
            system_prompt = """
                You are an AI agent with access to multiple MCP tools.

                For every user request:

                1. Check the available tools.
                2. Determine whether a tool can perform the requested task.
                3. If a suitable tool exists, use it.
                4. If multiple tools are needed, use them in the appropriate order.
                5. If no tool is required, answer the user directly.
                6. Do not return tool-generated code instead of executing the appropriate tool.

                Choose tools based on their descriptions and the user's request.

            """
    )

    query = HumanMessage("Create a Manim animation showing a circle transforming into a square. Use smooth animation and labels. Execute it with the Manim tool.")
    initial_state = {"messages": [query]}
    
    result = await agent.ainvoke(initial_state)

    print(result)

=== TOOLS ===
manim_execute_manim_code
manim_cleanup_manim_temp_dir
exchange_rate_get_conversion_factor
exchange_rate_convert
{'messages': [HumanMessage(content='Create a Manim animation showing a circle transforming into a square. Use smooth animation and labels. Execute it with the Manim tool.', additional_kwargs={}, response_metadata={}, id='2a43966f-7258-4733-ba25-5089541377e4'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants a Manim animation showing a circle transforming into a square, with smooth animation and labels. We need to use the manim_execute_manim_code tool. We need to provide the code. The code should create a scene that animates a circle morphing into a square. Use smooth animation, maybe using Transform or ReplacementTransform. Add labels. Then execute. We'll use the tool.", 'tool_calls': [{'id': 'fc_bf8c896f-d513-4b66-accd-a2093ebb8f7d', 'function': {'arguments': '{"manim_code":"from manim import *\\n\\nclass CircleToSquare(Scene):\\n 

In [22]:
async with MCPAdapter(clients) as adapter:
    tools = await adapter.list_tools()

    agent = create_agent(
        model=llm,
        tools=tools,
    )

    query = HumanMessage(''' Use the available FastCrawl tool to fetch https://www.zomato.com/ and return the page title and main text content.''')
    
    initial_state = {"messages": [query]}
    result = await agent.ainvoke(initial_state)
    last_message = result["messages"][-1]
    print(last_message.content)

**Page Title:**  
`Zomato`

**Main Text Content (summarized):**  
Zomato is India’s leading food‑delivery and restaurant‑discovery platform. It offers a fast, easy online ordering experience with over 300,000 restaurants in more than 800 cities and has delivered over 3 billion orders. The app includes features such as Veg Mode, healthy‑food collections, order scheduling, party planning, special offers, food delivery on trains, gourmet options, and gift cards. Zomato also promotes its Gold membership, which gives free delivery within 7 km and up to 30 % off at partner restaurants. The site encourages users to download the app for a seamless, feature‑rich food‑ordering experience.
